In [10]:
from sklearn.metrics import classification_report, roc_auc_score


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

df = pd.read_csv("../data/processed/telco_churn_clean.csv")
df.shape

(7043, 21)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Shape treino:", X_train_processed.shape)
print("Shape teste:", X_test_processed.shape)

Shape treino: (5634, 45)
Shape teste: (1409, 45)


In [3]:
# Converter para arrays densos (caso seja sparse) e depois para tensores float32
X_train_tensor = torch.tensor(np.asarray(X_train_processed), dtype=torch.float32)
X_test_tensor = torch.tensor(np.asarray(X_test_processed), dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

print(X_train_tensor.shape, y_train_tensor.shape)
print(X_train_tensor.dtype, y_train_tensor.dtype)

torch.Size([5634, 45]) torch.Size([5634, 1])
torch.float32 torch.float32


In [4]:
class ChurnMLP(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train_tensor.shape[1]
model = ChurnMLP(input_size=input_size)
print(model)

ChurnMLP(
  (network): Sequential(
    (0): Linear(in_features=45, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [5]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:", optimizer)


Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [6]:
from sklearn.model_selection import train_test_split as tts

# Dividir o treino atual em treino + validação (85/15)
X_train_final, X_val, y_train_final, y_val = tts(
    X_train_tensor, y_train_tensor, test_size=0.15, random_state=SEED,
    stratify=y_train_tensor.numpy()
)

print("Treino:", X_train_final.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test_tensor.shape)

Treino: torch.Size([4788, 45])
Validação: torch.Size([846, 45])
Teste: torch.Size([1409, 45])


In [7]:
from torch.utils.data import DataLoader, TensorDataset

# DataLoader: organiza os dados em batches e embaralha a cada época
train_dataset = TensorDataset(X_train_final, y_train_final)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Hiperparâmetros de treino
N_EPOCHS = 100
PATIENCE = 10  # quantas épocas sem melhora até parar

best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

train_losses = []
val_losses = []

for epoch in range(N_EPOCHS):
    # --- Fase de treino ---
    model.train()
    epoch_train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Fase de validação ---
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val).item()
    val_losses.append(val_loss)

    # --- Early stopping ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or patience_counter == 0:
        print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping na epoch {epoch+1} (sem melhora por {PATIENCE} epochs)")
        break

# Restaurar os melhores pesos encontrados
model.load_state_dict(best_model_state)
print(f"\nMelhor val_loss: {best_val_loss:.4f}")

Epoch 1: train_loss=0.5118, val_loss=0.4385
Epoch 2: train_loss=0.4342, val_loss=0.4310
Epoch 3: train_loss=0.4266, val_loss=0.4269
Epoch 4: train_loss=0.4210, val_loss=0.4216
Epoch 5: train_loss=0.4207, val_loss=0.4198
Epoch 6: train_loss=0.4177, val_loss=0.4188
Epoch 11: train_loss=0.4126, val_loss=0.4233

Early stopping na epoch 16 (sem melhora por 10 epochs)

Melhor val_loss: 0.4188


In [11]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_proba = torch.sigmoid(test_logits).numpy().flatten()
    test_pred = (test_proba >= 0.5).astype(int)

print(classification_report(y_test, test_pred, zero_division=0))
print("AUC-ROC:", roc_auc_score(y_test, test_proba))

              precision    recall  f1-score   support

           0       0.85      0.87      0.86      1035
           1       0.62      0.58      0.60       374

    accuracy                           0.79      1409
   macro avg       0.73      0.72      0.73      1409
weighted avg       0.79      0.79      0.79      1409

AUC-ROC: 0.8419282337440905


In [13]:
import mlflow
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
mlflow.set_experiment("churn-prediction-baselines")

<Experiment: artifact_location='file:///c:/dev/tech-challenge-fase1/notebooks/mlruns/1', creation_time=1781472055141, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781472055141, lifecycle_stage='active', name='churn-prediction-baselines', tags={}, trace_location=None, workspace='default'>

In [14]:
with mlflow.start_run(run_name="mlp_pytorch"):
    mlflow.log_param("model_type", "MLP")
    mlflow.log_param("architecture", "45-32-16-1")
    mlflow.log_param("activation", "ReLU")
    mlflow.log_param("dropout", 0.2)
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("early_stopping_patience", PATIENCE)
    mlflow.log_param("epochs_trained", epoch + 1)

    auc = roc_auc_score(y_test, test_proba)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("best_val_loss", best_val_loss)

print("Run da MLP registrada no MLflow.")

Run da MLP registrada no MLflow.
